# **＜重要＞本章で使用するファイル**

- real_estate_simple.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

In [ ]:
# japanize_matplotlib インストール
!pip install japanize_matplotlib

# Menu.49 データを準備しよう

基本情報を確認する

In [ ]:
import pandas as pd
df = pd.read_csv("real_estate_simple.csv")

print(df.head()) # 先頭5件を表示
print(df.info())

基本統計量を確認する

In [ ]:
print(df.describe())

面積と価格の散布図を作成する

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# 散布図の作成
plt.scatter(df["area"], df["price"])
plt.xlabel("面積(㎡ )")
plt.ylabel("価格(万円)")
plt.title("不動産の面積と価格の関係")
plt.grid(True, alpha=0.3)
plt.show()

# Menu.50 機械学習モデルを構築しよう

説明変数と目的変数を準備する

In [ ]:
import pandas as pd

df = pd.read_csv("real_estate_simple.csv")

X = df[["area"]] # 説明変数
y = df["price"] # 目的変数

print("説明変数(X)", X.shape)
print("目的変数(y)", y.shape)

データを訓練データとテストデータに分割する

In [ ]:
from sklearn.model_selection import train_test_split

# データを訓練用80%、テスト用20%に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # テストデータの割合
    random_state=42 # シャッフル結果を固定
)

print("訓練データ(X_train):", X_train.shape)
print("テストデータ(X_test):", X_test.shape)
print("訓練データ(y_train):", y_train.shape)
print("テストデータ(y_test):", y_test.shape)

基本統計量を比べる

In [ ]:
print("訓練データの基本統計量")
print(X_train.describe())
print("テストデータの基本統計量")
print(X_test.describe())

モデルの構築

In [ ]:
from sklearn.linear_model import LinearRegression

# モデルの構築
model = LinearRegression()
print(model)

モデルを学習させる

In [ ]:
# モデルの学習
model.fit(X_train, y_train)

パラメータを確認する

In [ ]:
print("傾き(a):", model.coef_)
print("切片(b):", model.intercept_)

訓練データで予測する

In [ ]:
# 訓練データで予測
y_train_pred = model.predict(X_train)

print(y_train_pred)

実際の価格を表示する

In [ ]:
# .values でnumpy配列に変換
print(y_train.values[:5]) # 先頭5件を取得

テストデータで実際の価格と予測値を比較する

In [ ]:
# テストデータで予測
y_test_pred = model.predict(X_test)

print("予測した価格:", y_test_pred[:5]) # 先頭5件を取得
print("実際の価格:", y_test.values[:5]) # 先頭5件を取得

実際の価格と予測した価格を散布図で比較する

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# テストデータの実際の値と予測値を散布図で比較
plt.figure(figsize=(10, 6))

# 実際のデータ点
plt.scatter(X_test, y_test, alpha=0.6, label="実際の価格")

# 予測値
plt.scatter(X_test, y_test_pred, alpha=0.6, label="予測した価格")

plt.xlabel("面積(㎡ )")
plt.ylabel("価格(万円)")
plt.title("テストデータ:実際の価格と予測した価格")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

MAEを計算する

In [ ]:
from sklearn.metrics import mean_absolute_error

# テストデータでMAEを計算
mae_test = mean_absolute_error(y_test, y_test_pred)

print("MAE (万円):", mae_test)

訓練データでMAEを計算する

In [ ]:
# 訓練データでMAEを計算
y_train_pred = model.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)

print("訓練データのMAE:", mae_train)

# Menu.51 構築したモデルを保存しよう

本番用モデルを構築する

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# データの読み込み
df = pd.read_csv("real_estate_simple.csv")
X = df[["area"]]
y = df["price"]

# 全データでモデルを学習(本番用)
model = LinearRegression()
model.fit(X, y)

print("傾き:", model.coef_)
print("切片:", model.intercept_)

学習済みモデルを保存する

In [ ]:
import pickle

# 学習済みモデルを保存
with open("real_estate_model.pkl", "wb") as f:
  pickle.dump(model, f)

Googleドライブにモデルを保存する

In [ ]:
# Googleドライブに保存する
with open("/content/drive/MyDrive/real_estate_model.pkl", "wb") as f:
    pickle.dump(model, f)

保存したモデルを読み込む

In [ ]:
# 保存したモデルを読み込む
with open("real_estate_model.pkl", "rb") as f:
  loaded_model = pickle.load(f)

# 読み込んだモデルのパラメータを確認
print("傾き(係数):", loaded_model.coef_)
print("切片:", loaded_model.intercept_)

未知の物件の価格を予測する

In [ ]:
# 新しい物件データ(面積のみ)
new_properties = pd.DataFrame({
    "area": [45, 60, 75, 90]
})

# 予測
predictions = loaded_model.predict(new_properties)
print("予測値", predictions)

## Column joblibによるモデルの保存

joblibでモデルを保存する

In [ ]:
import joblib

# 保存
joblib.dump(model, "real_estate_model.joblib")
# 読み込み
loaded_model = joblib.load("real_estate_model.joblib")